# 🎙️ Chatterbox-Bangla — Nijer Voice e Fine-tune (Colab T4)

Ei notebook `Banglabox/chatterbox-bangla-tts` model ke **tomar nijer voice** diye fine-tune (speaker adaptation) kore, jate model tomar gola te high-similarity **Bangla** speech banate pare.

**Age koro:** `Runtime` → `Change runtime type` → **T4 GPU** → Save. Tarpor cell gulo upor theke niche ek ek kore run koro.

| Step | Ki hoy |
|---|---|
| 1 | GPU check |
| 2 | Dependencies install (pinned versions) |
| 3 | Banglabox snapshot download + workspace assemble |
| 4 | Nijer voice audio upload (5-10 min) |
| 5 | Audio slice (silero-vad) + Bangla transcribe (faster-whisper) → LJSpeech dataset |
| 6 | Bangla-preserving base banao (released LoRA adapter merge) |
| 7 | Training run (T4 tuned, full T3 finetune via train.py) |
| 8 | Inference — nijer gola te Bangla text bolao |
| 9 | Trained model zip + download |

---
### ⚠️ Important — ei repo-r asol behavior (source theke pora, guess na):

- **`training/train.py` FULL T3 finetune kore** (LoRA na). Output = `chatterbox_output/t3_finetuned.safetensors` (puro T3 weight).
- Released `checkpoint/adapter/` holo ekta **LoRA adapter** (r=96, alpha=192, modules_to_save=text_emb+text_head). Kintu `train.py` ei adapter theke **init support kore na** — o base ResembleAI weight theke shuru kore, notun Bangla token gulo mean-init kore.
- Base ResembleAI weight Bangla jane **na** (Bangla knowledge shudhu LoRA adapter e ache). Tai shudhu 5-10 min data te base theke full-finetune korle **Bangla quality kharap hobe**.
- **Solution (Step 6):** released LoRA adapter ke resize-kora (vocab=2530) T3 er sathe **merge** kore ekta "Bangla-jana base" banai, tarpor training oi Bangla weight theke shuru hoy → Bangla thake, shudhu speaker adapt hoy. Eta repo-r nijer function diyei kora (`resize_and_load_t3_weights` + `PeftModel.merge_and_unload`, thik jemon `inference/infer.py` kore).
- **T4 e bf16 kaj kore na.** train.py te `bf16=True` hardcoded — amra runner e eta `fp16=True` te patch kori (niche).


## 1. GPU check
`T4` dekhale thik ache. `cpu` dekhale `Runtime > Change runtime type > T4 GPU` koro.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))
else:
    print('GPU nai! Runtime > Change runtime type > T4 GPU koro, tarpor abar run koro.')
!nvidia-smi -L

## 2. Dependencies install (pinned)

Training + inference dutar jonno known-working pinned stack: `transformers==4.46.3`, `tokenizers==0.20.3`, `peft==0.17.1`, `torch/torchaudio==2.6.0`. Plus dataset toirir jonno `faster-whisper` + Bangla text normalizer packages.

> ⏳ Install e 3-6 min lagte pare. **Colab restart chaite pare** (torch/numpy change hole). Restart chaile koro, tarpor Step 2 SKIP kore Step 3 theke continue koro.
> TODO: Colab ekhon Python 3.13. Kono wheel (torch 2.6 / ctranslate2) na pele runtime downgrade korte hobe.

In [ ]:
# training/requirements.txt-er pinned set install kori, tarpor known-good inference pins force kori
!pip -q install torch==2.6.0 torchaudio==2.6.0 transformers==4.46.3 tokenizers==0.20.3 peft==0.17.1 accelerate safetensors==0.5.3 librosa==0.11.0 soundfile==0.13.1 s3tokenizer==0.3.0 conformer==0.3.2 diffusers==0.29.0 einops==0.8.2 omegaconf==2.3.1 numpy==2.1.2 silero-vad==6.2.0 pyloudnorm num2words pandas tqdm tensorboard hf_transfer bnnumerizer bnunicodenormalizer bangla faster-whisper
print('Install done. Kono ERROR/restart-notice thakle porun.')

## 3. Banglabox snapshot download + workspace assemble

Training code HuggingFace repo `Banglabox/chatterbox-bangla-tts`-r `training/` folder e ache. Snapshot download kore writable jaigay assemble kori:
- `/content/cbtrain/` → training code (`train.py`, `src/`) + `pretrained_models/` (base weight + Bangla tokenizer 2530) + `checkpoint/` (LoRA adapter + NEW_VOCAB_SIZE.txt).
- `/content/cbinfer/` → inference code (`infer.py`, `bn_norm.py`, `src/`).

> Note: `train.py`-r `check_pretrained_models()` cwd-r `./pretrained_models` khoje, tai `pretrained_models` ke `cbtrain/` root e rakhi.

In [ ]:
import os, shutil
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import snapshot_download

REPO = 'Banglabox/chatterbox-bangla-tts'
snap = snapshot_download(REPO)   # public repo, token lage na
print('Snapshot:', snap)

ROOT = '/content/cbtrain'
INF  = '/content/cbinfer'
for p in (ROOT, INF):
    shutil.rmtree(p, ignore_errors=True)

# training/ er sob kichu ROOT root e (so `src` + train.py ROOT e boshe)
for item in os.listdir(f'{snap}/training'):
    s = f'{snap}/training/{item}'; d = f'{ROOT}/{item}'
    if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
    else: os.makedirs(ROOT, exist_ok=True); shutil.copy2(s, d)

# base weights + Bangla tokenizer, ar checkpoint (adapter + vocab size)
shutil.copytree(f'{snap}/pretrained_models', f'{ROOT}/pretrained_models', dirs_exist_ok=True)
shutil.copytree(f'{snap}/checkpoint',        f'{ROOT}/checkpoint',        dirs_exist_ok=True)

# inference env
shutil.copytree(f'{snap}/inference', INF, dirs_exist_ok=True)
# inference-r pretrained_models = training-r ta (duplicate na kore symlink)
if not os.path.exists(f'{INF}/pretrained_models'):
    os.symlink(f'{ROOT}/pretrained_models', f'{INF}/pretrained_models')
# example reference voices (chaile use korte paro)
shutil.copytree(f'{snap}/voices', '/content/voices_ref', dirs_exist_ok=True)

NEW_VOCAB = int(open(f'{ROOT}/checkpoint/NEW_VOCAB_SIZE.txt').read().strip())
ADAPTER_DIR = f'{ROOT}/checkpoint/adapter'
print('Workspace ready.')
print('NEW_VOCAB_SIZE =', NEW_VOCAB, '(config default 4240 na, eta 2530 — adapter er sathe match)')
print('Adapter        =', ADAPTER_DIR)
import json as _j
_tok = _j.load(open(f'{ROOT}/pretrained_models/tokenizer.json', encoding='utf-8'))
print('tokenizer vocab =', len(_tok['model']['vocab']), '| [bn] token:', '[bn]' in _tok['model']['vocab'])

## 4. Nijer voice audio upload

Tomar nijer **5-10 min** clean recording (`.wav` ba `.mp3`) upload koro. Ek ba multiple file dite paro. Jehetu eta Bangla model — **Bangla te kotha bola** recording dile similarity best hobe.

Tips: background noise kom, ek e speaker, consistent mic.

In [ ]:
from google.colab import files
import os, shutil
RAW = '/content/raw_audio'
shutil.rmtree(RAW, ignore_errors=True); os.makedirs(RAW, exist_ok=True)
print('Voice file(s) upload koro (.wav / .mp3)...')
up = files.upload()
for name in up:
    dst = os.path.join(RAW, os.path.basename(name))
    os.replace(name, dst)
    print('saved:', dst)
print('\nTotal uploaded:', len(os.listdir(RAW)), 'file')

## 5. Slice (silero-vad) + Bangla transcribe (faster-whisper) → LJSpeech dataset

`train.py`-r LJSpeech format (source `preprocess_ljspeech.py` theke pora):
- `wav_dir/` → chhoto `.wav` clip gulo
- `metadata.csv` → `sep="|"`, header optional. Code **column 0 = filename** (`.wav` na thakle add hoy, directory prefix strip hoy) ar **column 1 = text** use kore. (config comment "ID|RawText|NormText" bolleo code shudhu col0+col1 dhore.)
- Bangla text auto-normalize hoy preprocessing-e (bangla normalizer + punc_norm).

Ei cell:
1. Silero-VAD diye speech region ber kore, 4-12s clip e cut kore (natural pause e).
2. faster-whisper `large-v3` (`language='bn'`) diye protita clip transcribe kore.
3. `MyTTSDataset/wavs/` + `MyTTSDataset/metadata.csv` banay.

> ⏳ large-v3 first time ~3GB download + transcription-e kichu min lagbe.

In [ ]:
import os, shutil, csv, glob, math
import torch, torchaudio
import soundfile as sf
import numpy as np

DATA = '/content/MyTTSDataset'
WAVS = f'{DATA}/wavs'
shutil.rmtree(DATA, ignore_errors=True); os.makedirs(WAVS, exist_ok=True)

TARGET_SR   = 24000    # clip save sample rate (preprocess S3_SR e resample korbe, tao clean rakhi)
MIN_CLIP_S  = 4.0
MAX_CLIP_S  = 12.0
MERGE_GAP_S = 0.35     # ei gap er cheye chhoto pause hole segment merge

# ---- 1) silero-vad diye clip banai ----
from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
vad = load_silero_vad()

def slice_file(path, base):
    wav16 = read_audio(path, sampling_rate=16000)   # mono float tensor @16k
    ts = get_speech_timestamps(wav16, vad, sampling_rate=16000,
                               min_silence_duration_ms=250, threshold=0.5)
    if not ts:
        return []
    # gap chhoto hole merge
    merged = [dict(ts[0])]
    for seg in ts[1:]:
        if (seg['start'] - merged[-1]['end']) / 16000.0 <= MERGE_GAP_S:
            merged[-1]['end'] = seg['end']
        else:
            merged.append(dict(seg))
    # 4-12s window e greedily pack
    chunks, cur_s, cur_e = [], None, None
    for seg in merged:
        s, e = seg['start']/16000.0, seg['end']/16000.0
        if cur_s is None:
            cur_s, cur_e = s, e
        elif (e - cur_s) <= MAX_CLIP_S:
            cur_e = e
        else:
            if (cur_e - cur_s) >= MIN_CLIP_S: chunks.append((cur_s, cur_e))
            cur_s, cur_e = s, e
        if cur_e - cur_s >= MAX_CLIP_S:
            chunks.append((cur_s, cur_e)); cur_s, cur_e = None, None
    if cur_s is not None and (cur_e - cur_s) >= MIN_CLIP_S:
        chunks.append((cur_s, cur_e))

    # original file full-res load, resample TARGET_SR, mono
    wav, sr = torchaudio.load(path)
    if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.transforms.Resample(sr, TARGET_SR)(wav)
    wav = wav.squeeze(0).numpy()
    out = []
    for i, (s, e) in enumerate(chunks):
        a, b = int(s*TARGET_SR), int(e*TARGET_SR)
        clip = wav[a:b]
        if len(clip) < int(MIN_CLIP_S*TARGET_SR): continue
        name = f'{base}_{i:04d}.wav'
        sf.write(f'{WAVS}/{name}', clip, TARGET_SR)
        out.append(name)
    return out

all_clips = []
for f in sorted(glob.glob('/content/raw_audio/*')):
    base = os.path.splitext(os.path.basename(f))[0].replace(' ', '_')
    c = slice_file(f, base)
    print(f'{os.path.basename(f)} -> {len(c)} clips')
    all_clips += c
print('Total clips:', len(all_clips))

# ---- 2) faster-whisper diye Bangla transcribe ----
from faster_whisper import WhisperModel
wm = WhisperModel('large-v3', device='cuda' if torch.cuda.is_available() else 'cpu',
                  compute_type='float16' if torch.cuda.is_available() else 'int8')

rows = []
for name in all_clips:
    segs, _ = wm.transcribe(f'{WAVS}/{name}', language='bn', beam_size=5, vad_filter=False)
    text = ' '.join(s.text.strip() for s in segs).strip()
    if text:
        rows.append((name, text))

# whisper VRAM free
del wm
import gc; gc.collect(); torch.cuda.empty_cache()

# ---- 3) metadata.csv (pipe-separated, no header) ----
with open(f'{DATA}/metadata.csv', 'w', encoding='utf-8', newline='') as fp:
    w = csv.writer(fp, delimiter='|', quoting=csv.QUOTE_NONE, escapechar='\\')
    for name, text in rows:
        w.writerow([name, text])
print('metadata.csv written. Usable clips (non-empty text):', len(rows))

### Dataset review (recommended)
Transcription-e bhul thakte pare. Niche sample dekho. **Best result er jonno `MyTTSDataset/metadata.csv` manually check/correct kora bhalo** (bhul word thakle model bhul shikhbe).

> TODO (manual): Colab file browser (baame folder icon) → `MyTTSDataset/metadata.csv` open kore text গুলো thik koro. Format: `clipname.wav|bangla text`.

In [ ]:
import pandas as pd, soundfile as sf, glob, os
df = pd.read_csv('/content/MyTTSDataset/metadata.csv', sep='|', header=None, quoting=3,
                 names=['file', 'text'])
print('Rows:', len(df))
# total duration
dur = 0.0
for f in glob.glob('/content/MyTTSDataset/wavs/*.wav'):
    info = sf.info(f); dur += info.frames / info.samplerate
print(f'Total audio: {dur/60:.1f} min across {len(glob.glob("/content/MyTTSDataset/wavs/*.wav"))} clips')
print('\nSample rows:')
for _, r in df.head(8).iterrows():
    print(' ', r['file'], '->', str(r['text'])[:70])
if len(df) < 40:
    print('\n[!] Warning: 40-r kom clip. Beshi data (5-10 min) dile similarity onek bhalo hobe.')

## 6. Bangla-preserving base banao (released LoRA adapter merge)

Karon: `train.py` base ResembleAI weight theke shuru kore (Bangla jane na). Shudhu speaker adapt korle jate **Bangla thake**, tar jonno released Bangla LoRA adapter ke base er sathe merge kori — thik jemon `inference/infer.py` kore: `from_local` → `resize_and_load_t3_weights(vocab=2530)` → `PeftModel.from_pretrained(adapter)` → `merge_and_unload()`. Result = puro T3 (vocab 2530) je Bangla jane. Etake training er start weight hisebe use korbo (Step 7).

`KEEP_BANGLA=False` korle ei step skip kore plain base theke train hobe (Bangla quality kharap hobe — recommend kori na).

In [ ]:
import os, sys, torch
KEEP_BANGLA = True     # False korle plain base theke (Bangla harabe)
BANGLA_PT   = '/content/bangla_merged_t3.pt'

if KEEP_BANGLA:
    os.chdir('/content/cbtrain')                 # training src use kori (key match nishchit)
    sys.path.insert(0, '/content/cbtrain')
    from src.chatterbox_.tts import ChatterboxTTS
    from src.chatterbox_.models.t3.t3 import T3
    from src.model import resize_and_load_t3_weights
    from peft import PeftModel

    base = ChatterboxTTS.from_local('./pretrained_models', device='cpu')
    ps  = base.t3.state_dict()
    cfg = base.t3.hp
    cfg.text_tokens_dict_size = NEW_VOCAB
    if hasattr(cfg, 'use_cache'): cfg.use_cache = False
    nt = T3(hp=cfg)
    nt = resize_and_load_t3_weights(nt, ps)      # base copy + notun token mean-init
    print('vocab after resize:', nt.state_dict()['text_emb.weight'].shape)

    peft_m = PeftModel.from_pretrained(nt, ADAPTER_DIR, is_trainable=False)
    merged = peft_m.merge_and_unload()           # LoRA + text_emb/text_head fold-in
    sd = merged.state_dict()
    # sanity: raw T3 key gulo achhe kina
    assert 'text_emb.weight' in sd and sd['text_emb.weight'].shape[0] == NEW_VOCAB, sd['text_emb.weight'].shape
    torch.save(sd, BANGLA_PT)
    print('Bangla-merged base saved:', BANGLA_PT, '| keys:', len(sd))
    del base, nt, peft_m, merged, sd, ps
    import gc; gc.collect(); torch.cuda.empty_cache()
else:
    print('KEEP_BANGLA=False — Step 7 plain base theke train korbe.')

## 7. Training run (T4-tuned)

`train.py` ke direct chalai, kintu runtime e teenta jinis patch kori (source function reuse, notun API na):
1. **`TrainConfig`** — path, `new_vocab_size=2530`, ar T4-friendly hyperparams (`batch_size=1`, `grad_accum=16`, `num_epochs=12`). (train_modal.py-r monkeypatch pattern.)
2. **`TrainingArguments`** — `bf16=False, fp16=True` (T4 e bf16 nai).
3. **`resize_and_load_t3_weights`** — Step 6-r Bangla-merged weight overlay kore (KEEP_BANGLA hole), jate training Bangla theke shuru hoy.

Preprocessing (`.pt` toiri) `train.py`-r bhitorei hoy (`preprocess=True`).

> **Time (T4, ~5-10 min data, 12 epoch):** roughly **1.5-3 ghonta**. Kom korte `NUM_EPOCHS` komao.
> **OOM hole:** `NUM_EPOCHS` same rekhe `MAX_SPEECH_LEN` (e.g. 600) ba clip length komao; `batch_size` already 1. Preprocessing-e VE/S3Gen GPU te thake, tai training-e VRAM tight hote pare.

In [ ]:
%%writefile /content/cbtrain/run_finetune.py
import os, sys
sys.path.insert(0, os.getcwd())   # cwd = /content/cbtrain

NEW_VOCAB   = int(open('./checkpoint/NEW_VOCAB_SIZE.txt').read().strip())
DATA        = '/content/MyTTSDataset'
OUTPUT      = '/content/chatterbox_output'
KEEP_BANGLA = os.environ.get('KEEP_BANGLA', '1') == '1'
BANGLA_PT   = os.environ.get('BANGLA_PT', '/content/bangla_merged_t3.pt')
NUM_EPOCHS  = int(os.environ.get('NUM_EPOCHS', '12'))
BATCH_SIZE  = int(os.environ.get('BATCH_SIZE', '1'))
GRAD_ACCUM  = int(os.environ.get('GRAD_ACCUM', '16'))
MAX_SPEECH  = int(os.environ.get('MAX_SPEECH_LEN', '850'))

import torch
import train   # train.py — import korle torch.load patch + name-gulo train namespace e bind hoy

# --- 1) TrainConfig override (instance-level, train_modal.py style) ---
from src.config import TrainConfig
_ov = dict(
    model_dir='./pretrained_models',
    csv_path=f'{DATA}/metadata.csv',
    wav_dir=f'{DATA}/wavs',
    preprocessed_dir=f'{DATA}/preprocess',
    output_dir=OUTPUT,
    ljspeech=True, json_format=False, preprocess=True,
    is_turbo=False, is_inference=False,
    new_vocab_size=NEW_VOCAB,
    batch_size=BATCH_SIZE, grad_accum=GRAD_ACCUM, num_epochs=NUM_EPOCHS,
    learning_rate=5e-6, save_steps=200, save_total_limit=2,
    dataloader_num_workers=2, max_text_len=256, max_speech_len=MAX_SPEECH,
    prompt_duration=3.0,
)
_orig_init = TrainConfig.__init__
def _patched_init(self):
    _orig_init(self)
    for k, v in _ov.items(): setattr(self, k, v)
TrainConfig.__init__ = _patched_init

# --- 2) T4: bf16 -> fp16 ---
from transformers import TrainingArguments as _TA
def _TA_t4(**kw):
    kw['bf16'] = False; kw['fp16'] = True
    return _TA(**kw)
train.TrainingArguments = _TA_t4

# --- 3) Bangla-merged weight overlay (KEEP_BANGLA) ---
if KEEP_BANGLA and os.path.exists(BANGLA_PT):
    _orig_resize = train.resize_and_load_t3_weights
    def _resize_overlay(new_model, pretrained_sd):
        m = _orig_resize(new_model, pretrained_sd)           # base copy + mean-init
        sd = torch.load(BANGLA_PT, map_location='cpu')
        miss, unexp = m.load_state_dict(sd, strict=False)    # Bangla weight boshai
        print(f'[overlay] Bangla weights loaded. missing={len(miss)} unexpected={len(unexp)}')
        return m
    train.resize_and_load_t3_weights = _resize_overlay
    print('[overlay] KEEP_BANGLA ON — training Bangla weight theke shuru hobe.')
else:
    print('[overlay] KEEP_BANGLA OFF ba file nai — plain base theke shuru.')

print(f'Config: vocab={NEW_VOCAB} batch={BATCH_SIZE} grad_accum={GRAD_ACCUM} '
      f'epochs={NUM_EPOCHS} max_speech={MAX_SPEECH}')
train.main()

In [ ]:
import os
os.chdir('/content/cbtrain')
os.environ['KEEP_BANGLA']    = '1' if KEEP_BANGLA else '0'
os.environ['BANGLA_PT']      = BANGLA_PT
os.environ['NUM_EPOCHS']     = '12'
os.environ['BATCH_SIZE']     = '1'
os.environ['GRAD_ACCUM']     = '16'
os.environ['MAX_SPEECH_LEN'] = '850'
# training run (preprocess + train) — output: /content/chatterbox_output/t3_finetuned.safetensors
!cd /content/cbtrain && python run_finetune.py

In [ ]:
import glob
print('Output files:')
for f in sorted(glob.glob('/content/chatterbox_output/*')):
    print(' ', f)
assert os.path.exists('/content/chatterbox_output/t3_finetuned.safetensors'), \
    't3_finetuned.safetensors nai — training log e error dekho.'
print('\nOK: t3_finetuned.safetensors ready.')

## 8. Inference — nijer gola te Bangla text bolao

`train.py` puro T3 weight (`t3_finetuned.safetensors`) save kore — LoRA adapter na — tai released `infer.py`-r `BanglaTTS` (ja LoRA adapter load kore) সরাসরি use kora jabe na. Ekhane amra `infer.py`-r **same load path** follow kori, kintu adapter er bodole **finetuned full weight** load kori:
`from_local` → `resize(2530)` → `t3_finetuned.safetensors` load → `generate(...)` locked GEN params diye.

Reference clip = tomar nijer ekta clip (zero-shot cloning + finetune dutai combine hoy). Bangla text `bn_norm.for_tts` diye normalize hoy, `[bn]` prepend hoy.

In [ ]:
import os, sys, glob, torch, numpy as np
os.chdir('/content/cbinfer')
sys.path = [p for p in sys.path if p != '/content/cbtrain']
sys.path.insert(0, '/content/cbinfer')

from src.chatterbox_.tts import ChatterboxTTS
from src.chatterbox_.models.t3.t3 import T3
from src.model import resize_and_load_t3_weights
from safetensors.torch import load_file
try:
    from bn_norm import for_tts as stage2_norm
except Exception as e:
    print('bn_norm load fail (', e, ') -> raw text use hobe'); stage2_norm = lambda x: x

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
VOCAB = int(open('/content/cbtrain/checkpoint/NEW_VOCAB_SIZE.txt').read().strip())

base = ChatterboxTTS.from_local('/content/cbtrain/pretrained_models', device='cpu')
ps = base.t3.state_dict(); cfg = base.t3.hp
cfg.text_tokens_dict_size = VOCAB
if hasattr(cfg, 'use_cache'): cfg.use_cache = False
nt = T3(hp=cfg); nt = resize_and_load_t3_weights(nt, ps)
sd = load_file('/content/chatterbox_output/t3_finetuned.safetensors')
miss, unexp = nt.load_state_dict(sd, strict=False)
print('finetuned load: missing', len(miss), 'unexpected', len(unexp))
base.t3 = nt
base.t3.to(DEV).eval(); base.s3gen.to(DEV).eval(); base.ve.to(DEV).eval()
base.device = DEV
SR = base.sr
GEN = dict(temperature=0.5, repetition_penalty=1.5, min_p=0.05, cfg_weight=0.5, exaggeration=0.5)
print('Model ready. sr =', SR)

In [ ]:
from IPython.display import Audio, display
import soundfile as sf, glob, numpy as np, torch

# reference = tomar nijer ekta clip (chaile onno clean clip er path dao)
REF = sorted(glob.glob('/content/MyTTSDataset/wavs/*.wav'))[0]
print('Reference clip:', REF)

TEXT = "আজকে আবহাওয়া খুব সুন্দর, চলো একটু ঘুরে আসি।"   # <-- nijer Bangla text likho

t = stage2_norm(TEXT)
t = t if t.strip().startswith('[bn]') else '[bn]' + t.strip()
with torch.no_grad():
    wav = base.generate(text=t, audio_prompt_path=REF, **GEN)
if isinstance(wav, tuple): wav = wav[0]
wav = wav.squeeze().cpu().numpy().astype('float32')
sf.write('/content/output.wav', wav, SR)
print(f'Generated {len(wav)/SR:.2f}s -> /content/output.wav')
display(Audio(wav, rate=SR))

## 9. Trained model zip + download

Reuse er jonno finetuned weight (+ dorkari kit file) zip kore download koro. Ei `t3_finetuned.safetensors` + `pretrained_models` (Bangla tokenizer soho) + `NEW_VOCAB_SIZE.txt` diye por e Step 8-r loader chala jabe.

In [ ]:
import shutil, os
KIT = '/content/finetuned_kit'
shutil.rmtree(KIT, ignore_errors=True); os.makedirs(KIT, exist_ok=True)
shutil.copy2('/content/chatterbox_output/t3_finetuned.safetensors', f'{KIT}/t3_finetuned.safetensors')
shutil.copy2('/content/cbtrain/checkpoint/NEW_VOCAB_SIZE.txt', f'{KIT}/NEW_VOCAB_SIZE.txt')
# tokenizer.json (Bangla 2530) — reuse er jonno dorkari
shutil.copy2('/content/cbtrain/pretrained_models/tokenizer.json', f'{KIT}/tokenizer.json')
shutil.make_archive('/content/finetuned_kit', 'zip', KIT)
print('Zip:', '/content/finetuned_kit.zip', '(', round(os.path.getsize("/content/finetuned_kit.zip")/1e6,1), 'MB )')
from google.colab import files
files.download('/content/finetuned_kit.zip')